# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/real-huzaifa/flyrank-internship-ml-track/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am taking **Lane 2: Refresh / Content Opportunity Scoring**.

I chose it because the decision it serves is real and capacity-bound. A content team cannot
review 30,000 pages; it reviews a few dozen a week. The cell below shows that simple,
transparent triggers already flag **13,191 pages** — more than five years of review capacity at
50 pages per week. That tells me detection is not the hard part. **Ordering is.** A lane whose
output is a ranked queue therefore attacks the part of the problem that is actually binding.

I also chose it with eyes open about the starter pipeline, which implements a version of this
lane already. Its label (`is_declining_label = trend_direction == "down"`) is a *current-window
bucket*, not a future outcome — the data guide calls this a beginner proxy label. So I am
treating the starter pipeline as **the baseline I have to beat**, not as the project. My
intended contribution is a genuine future-window label (features from the prior 90 days,
decline measured over the next 30 days) built from the warehouse daily facts, with a leakage
audit, once warehouse access is working in Week 3.

In [1]:
import os
from pathlib import Path
import pandas as pd

REPO = "flyrank-internship-ml-track"

# Colab loads only the .ipynb from GitHub, not the repo — clone if the data is missing.
# Running locally from inside the repo, this block is skipped.
if not Path("data/raw").exists():
    if not Path(REPO).exists():
        !git clone https://github.com/real-huzaifa/{REPO}.git
    os.chdir(REPO)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns | {df['client_id'].nunique()} clients")
print()

# The number that decided my lane: candidate volume vs. review capacity
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]

candidates = (
    ((eligible["days_since_last_update"] >= 180) & (eligible["impressions_90d"] >= 500)) |
    ((eligible["trend_direction"] == "down") & (eligible["impressions_90d"] >= 100)) |
    ((eligible["word_count"] > 0) & (eligible["word_count"] < 1200) & (eligible["impressions_90d"] >= 250))
).sum()

WEEKLY_CAPACITY = 50
print(f"Pages flagged by simple triggers: {candidates:,}")
print(f"At {WEEKLY_CAPACITY} reviews/week: {candidates/WEEKLY_CAPACITY:.0f} weeks "
      f"(~{candidates/WEEKLY_CAPACITY/52:.1f} years)")
print()
print("=> Detection is solved. Ordering is not.")

Cloning into 'flyrank-internship-ml-track'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 119 (delta 35), reused 94 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.84 MiB | 13.79 MiB/s, done.
Resolving deltas: 100% (35/35), done.
Loaded: 30,000 rows x 44 columns | 32 clients

Pages flagged by simple triggers: 13,191
At 50 reviews/week: 264 weeks (~5.1 years)

=> Detection is solved. Ordering is not.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among content pages with existing search demand, which ones should a
content team review first — and can a learned ranking order that queue better than the
transparent rule the team would otherwise write by hand?

### The frame, in one paragraph

For a content review team with limited weekly capacity, deciding which pages to review first
for refresh, we will build a ranked review queue from observed search and content signals in
the starter dataset (impressions, clicks, average position, age, freshness, word count,
engagement), scoring each page by its probability of sustained traffic decline, measured by
precision@50 under client-grouped validation plus a by-hand read of the top 20. A wrong call
costs an editor's hour spent on a page that was fine; a missed call leaves a genuinely
declining page unattended. A plain rule is not enough because the transparent triggers already
flag 13,191 of 30,000 pages — over five years of review capacity at 50 pages a week — so
detection is already solved and *ordering* is the real problem. We will claim only observed,
decision-support results.

### The four framing questions, answered

**1. What decision does this improve?**  
Which pages enter this week's review queue, and in what order. Not "predict decline" — the
decision is the allocation of a fixed number of editor-hours across a much larger candidate
pool.

**2. Who acts, and what do they do?**  
A content editor works down the ranked list. For each page they either perform a refresh action
(update, expand, consolidate, or prune) or mark it "no action needed" with a reason. The reason
codes attached to each ranked page tell them *why* it surfaced, so they can disagree with the
ranking quickly instead of re-deriving it.

**3. What does a wrong answer cost — and which error is worse?**  
A false positive costs roughly one editor-hour on a page that did not need attention. A false
negative leaves a genuinely declining page unreviewed, and traffic keeps eroding until the next
cycle. Because the queue is capacity-bound and the team only ever sees the top of the list,
**precision at the top matters more than total recall**. I would rather hand over 50 solid
candidates than 500 mixed ones. This is why the headline metric is precision@50 and not
accuracy — as the cell below shows, the base rate makes accuracy close to meaningless here.

**4. Why does data or ML help at all?**  
Because a plain rule does not fail by being wrong — it fails by being *undiscriminating*. The
three transparent triggers flag 13,191 pages with no ordering among them, which is close to
handing the team a random subset. The signal separating "declining and worth an hour" from
"declining and not worth it" is spread across many weak, interacting columns — demand,
position, age, freshness, depth, engagement — and shifts over time. That is precisely where a
learned ranking earns its place. If a well-tuned transparent score turns out to rank just as
well, that is a real result and I will report it as one rather than bury it.

### Task type

| Element | Choice |
|---|---|
| Task type | Ranking / scoring ("which ones first?") |
| Target | Probability of sustained traffic decline |
| Metric | precision@50, plus average precision; by-hand review of top 20 |
| Validation | Client-grouped holdout (pages from one client must not span train and test) |

In [2]:
# Why precision@K and not accuracy: check the base rate
label = (eligible["trend_direction"] == "down")

print(f"Base rate (trend_direction == 'down'): {label.mean()*100:.1f}%")
print(f"  positives = {label.sum():,}   negatives = {(~label).sum():,}")
print()
print(f"A model that predicts 'declining' for EVERY page scores {label.mean()*100:.1f}% accuracy")
print("and is worthless to the team. Hence precision@50 as the headline metric.")

Base rate (trend_direction == 'down'): 54.2%
  positives = 16,262   negatives = 13,738

A model that predicts 'declining' for EVERY page scores 54.2% accuracy
and is worthless to the team. Hence precision@50 as the headline metric.


## 3. Quick look at the data (2-3 real numbers)

Three numbers to justify spending the next seven weeks on this lane:

1. how much of the slice is actually in play,
2. the base rate of the starter label — and why it rules out accuracy as a metric,
3. the gap between candidate volume and review capacity, which is the case for the lane.

In [3]:
# [1] Eligible set, using the starter pipeline's own filters
print(f"[1] Eligible pages: {len(eligible):,} of {len(df):,}")
print(f"    Filters: impressions_90d > 0 AND content_age_days >= 90")
print(f"    Removed: {len(df) - len(eligible):,} rows")
print()

# [2] Base rate of the starter proxy label
print(f"[2] Base rate (trend_direction == 'down'): {label.mean()*100:.1f}%")
print(f"    positives = {label.sum():,}   negatives = {(~label).sum():,}")
print()
print("    Full trend_direction distribution:")
print(eligible["trend_direction"].value_counts().to_string())
print()

# [3] The capacity mismatch — the case for this lane
stale     = (eligible["days_since_last_update"] >= 180) & (eligible["impressions_90d"] >= 500)
declining = (eligible["trend_direction"] == "down")     & (eligible["impressions_90d"] >= 100)
thin      = (eligible["word_count"] > 0) & (eligible["word_count"] < 1200) & (eligible["impressions_90d"] >= 250)
any_trigger = stale | declining | thin

print("[3] Pages meeting simple review triggers:")
print(f"    stale_visible_page    : {stale.sum():,}")
print(f"    declining_with_demand : {declining.sum():,}")
print(f"    thin_visible_page     : {thin.sum():,}")
print(f"    ANY of the three      : {any_trigger.sum():,}")
print()
print(f"    At {WEEKLY_CAPACITY} reviews/week: {any_trigger.sum()/WEEKLY_CAPACITY:.0f} weeks "
      f"(~{any_trigger.sum()/WEEKLY_CAPACITY/52:.1f} years)")
print()
print("    => Detection is not the bottleneck. Ordering is.")

[1] Eligible pages: 30,000 of 30,000
    Filters: impressions_90d > 0 AND content_age_days >= 90
    Removed: 0 rows

[2] Base rate (trend_direction == 'down'): 54.2%
    positives = 16,262   negatives = 13,738

    Full trend_direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

[3] Pages meeting simple review triggers:
    stale_visible_page    : 17
    declining_with_demand : 13,152
    thin_visible_page     : 82
    ANY of the three      : 13,191

    At 50 reviews/week: 264 weeks (~5.1 years)

    => Detection is not the bottleneck. Ordering is.


## 4. Careful words: what I can and can't claim

### What these three numbers tell me

**[1]** The starter's own filters remove nothing on this slice — all 30,000 rows already have
search demand and are at least 90 days old. I am not working with a thin remainder; the whole
slice is in play.

**[2]** A 54.2% base rate is well balanced, which is convenient for modelling but also a
warning: **accuracy is a useless metric here.** A model that predicts "declining" for
everything scores 54% and helps nobody. This is a direct argument for precision@K, which
measures the only thing the team ever experiences — the quality of the top of the list.

**[3]** This is the number that justifies the lane. Simple triggers flag 13,191 pages and give
no ordering among them. A team reviewing 50 a week would need roughly five years to clear that
queue. The rule has already solved detection and left the actual problem untouched.

### Can claim (observed / directional)

- That within this 30,000-row anonymized slice, over a trailing 90-day window across 32
  clients, certain observable signals are *associated with* pages bucketed as declining.
- That simple transparent triggers flag a candidate pool far larger than a review team's
  capacity — this is arithmetic, not inference.
- That a learned ranking does or does not order that pool better than a transparent rule,
  under a stated validation design and a stated metric.

### Cannot claim

- **That refreshing a page causes recovery.** This is observational data with no experiment and
  no control group. Establishing causation would need a proper causal design. I will not make
  this claim anywhere in my write-up.
- **Anything about Google's ranking algorithm.** I observe outcomes, not mechanisms.
- **That results here generalise to the full warehouse.** This is a 30k starter slice; the
  warehouse is ~79M daily rows across 104 clients. Any result has to be re-earned there.

### Leakage found during the signal audit

The starter label chain is:

`impressions_last_30d` + `impressions_prev_30d` → `trend_pct` → `trend_direction` → `is_declining_label`

The cell below verifies that `trend_pct` is *exactly* the percentage change between the two
30-day impression windows. The data guide names `trend_direction` and `trend_pct` as forbidden
features — but it does **not** name the two raw columns they are computed from. Those are
equally unusable as features for this label, and I will exclude all four.

### Label quality caveat

Pages marked `flat` or `new` have a null `trend_pct` — their trend is *unknown*, not *not
declining* — yet `is_declining_label` counts them as negatives. Any baseline number I quote
inherits this flaw, and I will say so whenever I quote one.

### Two data traps I am carrying forward

- `avg_position = 0` means "no data", not rank zero. Every one of those rows is labelled
  `position_tier = top_3`, because 0 sorts below 3 — so any naive analysis by position tier
  silently contaminates the best tier with pages that have no position data at all.
- Rate columns (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) are ×100
  percentages, and `scroll_rate` / `ai_traffic_pct` can legitimately exceed 100 because their
  numerator and denominator come from different measurement systems.

### The honest limit of the starter label

`is_declining_label` describes a window that has already closed. It is a *description*, not a
*prediction* — a model trained on it learns to recognise a bucket, not to anticipate an
outcome. Acceptable as a Week 1 baseline, unacceptable as a capstone target. Hence the stated
upgrade: features from the prior 90 days, outcome measured over the following 30 days, windows
strictly non-overlapping and audited for leakage.

In [4]:
# SIGNAL AUDIT — is the starter label reconstructable from other columns?
audit = df[(df["impressions_prev_30d"] > 0) & df["trend_pct"].notna()].copy()
audit["reconstructed"] = (
    (audit["impressions_last_30d"] - audit["impressions_prev_30d"])
    / audit["impressions_prev_30d"] * 100
)
exact = (audit["trend_pct"] - audit["reconstructed"]).abs() < 0.5

print("LEAKAGE CHECK")
print(f"  rows tested          : {len(audit):,}")
print(f"  reconstructed exactly: {exact.sum():,} ({exact.mean()*100:.1f}%)")
print(f"  correlation          : {audit['trend_pct'].corr(audit['reconstructed']):.4f}")
print()
print("  trend_direction is just a threshold on trend_pct:")
print(df.groupby("trend_direction")["trend_pct"].agg(["min", "max", "count"]).round(1).to_string())
print()
print("  => impressions_last_30d and impressions_prev_30d are ALSO leakage for this label.")
print()

# LABEL QUALITY — what happens to pages whose trend cannot be computed?
unknown = df["trend_direction"].isin(["flat", "new"])
print("LABEL QUALITY CHECK")
print(f"  pages marked 'flat' or 'new': {unknown.sum():,} ({unknown.mean()*100:.1f}%)")
print(f"  of those, trend_pct is null : {df.loc[unknown, 'trend_pct'].isna().sum():,}")
print("  => trend is UNKNOWN, but the starter label files them as negatives.")
print()

# DOCUMENTED TRAP — avg_position == 0 means 'no data', not rank 0
zero_pos = df["avg_position"] == 0
print("POSITION TIER TRAP")
print(f"  avg_position == 0 rows: {zero_pos.sum():,}")
print(f"  their position_tier   : {df.loc[zero_pos, 'position_tier'].unique().tolist()}")
print("  => 'no data' pages are silently pooled into the best tier.")

LEAKAGE CHECK
  rows tested          : 26,612
  reconstructed exactly: 26,612 (100.0%)
  correlation          : 1.0000

  trend_direction is just a threshold on trend_pct:
                   min      max  count
trend_direction                       
down            -100.0    -20.0  16262
flat               NaN      NaN      0
new                NaN      NaN      0
stable           -20.0     20.0   5962
up                20.0  44900.0   4388

  => impressions_last_30d and impressions_prev_30d are ALSO leakage for this label.

LABEL QUALITY CHECK
  pages marked 'flat' or 'new': 3,388 (11.3%)
  of those, trend_pct is null : 3,388
  => trend is UNKNOWN, but the starter label files them as negatives.

POSITION TIER TRAP
  avg_position == 0 rows: 1,205
  their position_tier   : ['top_3']
  => 'no data' pages are silently pooled into the best tier.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.